# Classificação de Gliomas Cerebrais com Dados Clínicos e Moleculares

**Disciplina:** Sistemas Inteligentes  
**Dataset:** TCGA Glioma — [UCI Repository ID 759](https://archive.ics.uci.edu/dataset/759/glioma+grading+clinical+and+mutation+features+dataset)  
**Referência base:** Tasci et al. (2022) — *Hierarchical Voting-Based Feature Selection and Ensemble Learning Model Scheme for Glioma Grading with Clinical and Molecular Characteristics*. Int. J. Mol. Sci.

## Objetivo
Avaliar a adequação de três modelos de aprendizado de máquina — **KNN**, **MLP** e **BernoulliNB** — para a classificação do grau de gliomas (LGG vs GBM) a partir de dados clínicos e moleculares do dataset TCGA, com análise dos padrões de erro em relação às features moleculares mais relevantes.

---
# 1. Instalação e Importações

In [ ]:
!pip install ucimlrepo -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display
from ucimlrepo import fetch_ucirepo

# Pré-processamento e pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

# Validação
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate, GridSearchCV
)

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import BernoulliNB

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, make_scorer
)

# Semente global para reprodutibilidade
SEED = 42
np.random.seed(SEED)

print('Bibliotecas carregadas com sucesso!')

---
# 2. Carregamento do Dataset

In [ ]:
# Carregando o dataset TCGA Glioma do UCI Repository (ID=759)
base_glioma = fetch_ucirepo(id=759)

X = base_glioma.data.features.copy()
y = base_glioma.data.targets.copy()

print('Dataset carregado com sucesso!')
print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

---
# 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Visão geral do dataset
print('=== Primeiras linhas ===')
display(X.head())
print()
print('=== Tipos de dados ===')
print(X.dtypes)
print()
print('=== Shape ===')
print(f'Instâncias: {X.shape[0]} | Features: {X.shape[1]}')

In [ ]:
# Estatísticas descritivas — útil especialmente para Age_at_diagnosis
print('=== Estatísticas Descritivas ===')
display(X.describe(include='all'))

In [ ]:
# Valores nulos por coluna
nulos = X.isnull().sum()
print('=== Valores Nulos por Coluna ===')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Nenhum valor nulo encontrado.')
print(f'\nTotal de nulos: {nulos.sum()}')

In [ ]:
# Valores duplicados
dup = X.duplicated().sum()
print(f'Linhas duplicadas: {dup}')

In [ ]:
# Distribuição das classes
contagem = y['Grade'].value_counts()
proporcao = y['Grade'].value_counts(normalize=True)

print('=== Distribuição das Classes ===')
print('Contagem:')
print(contagem)
print('\nProporção:')
print(proporcao.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(contagem.index, contagem.values,
              color=['#2196F3', '#F44336'], edgecolor='black', linewidth=0.8)
ax.set_title('Distribuição das Classes — LGG vs GBM', fontsize=13, fontweight='bold')
ax.set_xlabel('Grau do Glioma')
ax.set_ylabel('Número de Instâncias')
for bar, val in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identificação do perfil das features
COLUNAS_CONTINUAS   = ['Age_at_diagnosis']
COLUNAS_CATEGORICAS = ['Gender', 'Race']
COLUNAS_BINARIAS    = [col for col in X.columns
                       if col not in COLUNAS_CONTINUAS + COLUNAS_CATEGORICAS]

print('=== Perfil das Features ===')
print(f'Contínuas  ({len(COLUNAS_CONTINUAS)}):    {COLUNAS_CONTINUAS}')
print(f'Categóricas ({len(COLUNAS_CATEGORICAS)}): {COLUNAS_CATEGORICAS}')
print(f'Binárias   ({len(COLUNAS_BINARIAS)}):  {COLUNAS_BINARIAS}')
print()
print('Nota: As 20 features moleculares binárias representam o status de mutação')
print('(mutated / not_mutated) e NÃO serão normalizadas, pois já estão na mesma escala.')

In [ ]:
# Frequência de mutações nas features moleculares
X_temp = X.copy()
X_temp['Grade'] = y['Grade'].values

# Converter binárias para 0/1 para análise
for col in COLUNAS_BINARIAS:
    X_temp[col] = (X_temp[col] == 'Mutant').astype(int)

freq_mutacoes = X_temp[COLUNAS_BINARIAS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
freq_mutacoes.plot(kind='bar', ax=ax, color='#5C6BC0', edgecolor='black', linewidth=0.6)
ax.set_title('Frequência de Mutação por Gene no Dataset TCGA', fontsize=13, fontweight='bold')
ax.set_xlabel('Gene')
ax.set_ylabel('Frequência (proporção de casos)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('frequencia_mutacoes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Taxa de mutação por classe (LGG vs GBM) para features moleculares chave
features_chave = ['IDH1', 'TP53', 'ATRX', 'PTEN', 'EGFR']

taxa_por_classe = {}
for feat in features_chave:
    taxa_por_classe[feat] = X_temp.groupby('Grade')[feat].mean()

df_taxa = pd.DataFrame(taxa_por_classe).T
print('=== Taxa de Mutação por Classe (features chave) ===')
display(df_taxa.round(3))

df_taxa.plot(kind='bar', figsize=(10, 4), color=['#2196F3', '#F44336'],
             edgecolor='black', linewidth=0.6)
plt.title('Taxa de Mutação por Classe — Features Moleculares Chave', fontsize=13, fontweight='bold')
plt.xlabel('Gene')
plt.ylabel('Taxa de Mutação')
plt.xticks(rotation=0)
plt.legend(title='Grau')
plt.tight_layout()
plt.savefig('taxa_mutacao_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 4. Pré-processamento

In [ ]:
# Remover instâncias com valores ausentes (Gender, Age_at_diagnosis, Race)
# conforme descrito no artigo base (Tasci et al., 2022)
X_clean = X.copy()
y_clean = y['Grade'].copy()

# Tratar valores ausentes/inválidos nas colunas clínicas
for col in ['Gender', 'Race']:
    mascara_invalido = X_clean[col].isin(['--', 'not reported', 'Not Reported', ''])
    X_clean = X_clean[~mascara_invalido]
    y_clean = y_clean[X_clean.index]

# Remover nulos em Age_at_diagnosis
mascara_nulo = X_clean['Age_at_diagnosis'].isnull()
X_clean = X_clean[~mascara_nulo]
y_clean = y_clean[X_clean.index]

# Converter Age_at_diagnosis para float
X_clean['Age_at_diagnosis'] = pd.to_numeric(X_clean['Age_at_diagnosis'], errors='coerce')
mascara_nulo2 = X_clean['Age_at_diagnosis'].isnull()
X_clean = X_clean[~mascara_nulo2]
y_clean = y_clean[X_clean.index]

X_clean = X_clean.reset_index(drop=True)
y_clean = y_clean.reset_index(drop=True)

print(f'Shape após limpeza: {X_clean.shape}')
print(f'Instâncias removidas: {X.shape[0] - X_clean.shape[0]}')
print(f'\nDistribuição final das classes:')
print(y_clean.value_counts())

In [ ]:
# Codificar o rótulo: LGG=0, GBM=1
le = LabelEncoder()
y_encoded = le.fit_transform(y_clean)

print(f'Classes codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print('LGG=0 (negativo), GBM=1 (positivo — classe de interesse clínico)')

In [ ]:
# Definição do preprocessador
# - Age_at_diagnosis: Z-Score (StandardScaler) — única feature contínua
# - Gender, Race: OneHotEncoder — features categóricas
# - 20 features moleculares binárias: passthrough — já estão em escala {0,1}

# Garantir que as binárias estejam como 0/1
for col in COLUNAS_BINARIAS:
    X_clean[col] = (X_clean[col] == 'Mutant').astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        ('continuas',   StandardScaler(),
                        COLUNAS_CONTINUAS),
        ('categoricas', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
                        COLUNAS_CATEGORICAS),
        ('binarias',    'passthrough',
                        COLUNAS_BINARIAS)
    ],
    remainder='drop'
)

print('Preprocessador definido:')
print('  - Age_at_diagnosis → Z-Score (StandardScaler)')
print('  - Gender, Race     → OneHotEncoder (drop=first)')
print('  - 20 genes binários → passthrough (sem normalização)')

---
# 5. Divisão dos Dados

In [ ]:
# Hold-Out 70/30 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=SEED
)

print('=== Divisão Hold-Out (70/30) ===')
print(f'Treino: {X_train.shape[0]} instâncias')
print(f'  LGG: {(y_train == 0).sum()} | GBM: {(y_train == 1).sum()}')
print(f'Teste:  {X_test.shape[0]} instâncias')
print(f'  LGG: {(y_test == 0).sum()} | GBM: {(y_test == 1).sum()}')

# K-Fold 5 estratificado — aplicado sobre o treino durante GridSearch
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'\nK-Fold: {kfold.n_splits} folds estratificados (usado no GridSearchCV)')

---
# 6. Funções Auxiliares

In [ ]:
# Scorer customizado para GridSearch (F1 como critério principal)
scoring_cv = {
    'acuracia':  'accuracy',
    'precisao':  'precision',
    'recall':    'recall',
    'f1':        'f1',
    'kappa':     make_scorer(cohen_kappa_score),
    'auc':       'roc_auc'
}

def exibir_resultados_cv(scores, nome_modelo):
    """Exibe resultados do cross_validate com média ± desvio."""
    print(f'\n=== {nome_modelo} — K-Fold (5 folds) ===')
    print('-' * 45)
    for metrica in ['acuracia', 'precisao', 'recall', 'f1', 'kappa', 'auc']:
        vals = scores[f'test_{metrica}']
        print(f'{metrica.capitalize():12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')
    print('-' * 45)

def exibir_resultados_holdout(y_true, y_pred, y_proba, nome_modelo):
    """Exibe resultados no conjunto de teste (Hold-Out)."""
    print(f'\n=== {nome_modelo} — Hold-Out (30%) ===')
    print('-' * 45)
    print(f'Acurácia  : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precisão  : {precision_score(y_true, y_pred):.4f}')
    print(f'Recall    : {recall_score(y_true, y_pred):.4f}')
    print(f'F1        : {f1_score(y_true, y_pred):.4f}')
    print(f'Kappa     : {cohen_kappa_score(y_true, y_pred):.4f}')
    print(f'AUC-ROC   : {roc_auc_score(y_true, y_proba):.4f}')
    print('-' * 45)

def construir_pipeline(modelo):
    """Constrói um pipeline com preprocessador + modelo."""
    return Pipeline([
        ('preprocessamento', preprocessor),
        ('modelo', modelo)
    ])

def executar_grid(modelo, params, nome):
    """Executa GridSearchCV e retorna o melhor estimador."""
    pipeline = construir_pipeline(modelo)
    params_pipeline = {f'modelo__{k}': v for k, v in params.items()}

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=params_pipeline,
        cv=kfold,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    print(f'\n=== {nome} — Melhores Hiperparâmetros ===')
    for k, v in grid.best_params_.items():
        print(f'  {k.replace("modelo__", "")}: {v}')
    print(f'  Melhor F1 no CV: {grid.best_score_:.4f}')

    return grid.best_estimator_

print('Funções auxiliares definidas!')

---
# 7. Treinamento dos Modelos

## Justificativa da escolha dos modelos

O dataset TCGA possui um perfil de features muito específico: **20 features moleculares binárias** (mutado/não mutado), **1 feature contínua** (idade) e **2 features categóricas** (gênero, raça). Essa composição mista guia a escolha dos modelos:

- **KNN**: baseline interpretável; a similaridade entre pacientes reflete o padrão combinado de mutações. Testamos a distância de Hamming, matematicamente mais adequada para dados binários.
- **MLP**: capaz de aprender interações não-lineares entre mutações, capturando efeitos sinérgicos entre genes.
- **BernoulliNB**: especificamente projetado para features binárias; modela a probabilidade de cada mutação estar presente dado o grau do tumor.

## 7.1 KNN

In [ ]:
params_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan', 'hamming']  # hamming: adequado para binários
}

melhor_knn = executar_grid(KNeighborsClassifier(), params_knn, 'KNN')

In [ ]:
# Avaliar KNN com cross_validate (K-Fold) sobre o melhor modelo
scores_knn_cv = cross_validate(
    melhor_knn, X_train, y_train,
    cv=kfold, scoring=scoring_cv
)
exibir_resultados_cv(scores_knn_cv, 'KNN')

In [ ]:
# Treinar o melhor KNN no treino completo e avaliar no Hold-Out
melhor_knn.fit(X_train, y_train)
y_pred_knn   = melhor_knn.predict(X_test)
y_proba_knn  = melhor_knn.predict_proba(X_test)[:, 1]
exibir_resultados_holdout(y_test, y_pred_knn, y_proba_knn, 'KNN')

## 7.2 MLP

In [ ]:
params_mlp = {
    'hidden_layer_sizes': [(32,), (64,), (32, 16), (64, 32)],
    'activation':         ['relu', 'tanh'],
    'solver':             ['adam', 'lbfgs'],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate':      ['constant', 'adaptive']
}

melhor_mlp = executar_grid(
    MLPClassifier(random_state=SEED, early_stopping=True, max_iter=500),
    params_mlp, 'MLP'
)

In [ ]:
scores_mlp_cv = cross_validate(
    melhor_mlp, X_train, y_train,
    cv=kfold, scoring=scoring_cv
)
exibir_resultados_cv(scores_mlp_cv, 'MLP')

In [ ]:
melhor_mlp.fit(X_train, y_train)
y_pred_mlp   = melhor_mlp.predict(X_test)
y_proba_mlp  = melhor_mlp.predict_proba(X_test)[:, 1]
exibir_resultados_holdout(y_test, y_pred_mlp, y_proba_mlp, 'MLP')

## 7.3 BernoulliNB

In [ ]:
params_bnb = {
    'alpha':      [1e-9, 1e-3, 0.1, 0.5, 1.0],
    'binarize':   [None, 0.0, 0.5],
    'fit_prior':  [True, False]
}

melhor_bnb = executar_grid(BernoulliNB(), params_bnb, 'BernoulliNB')

In [ ]:
scores_bnb_cv = cross_validate(
    melhor_bnb, X_train, y_train,
    cv=kfold, scoring=scoring_cv
)
exibir_resultados_cv(scores_bnb_cv, 'BernoulliNB')

In [ ]:
melhor_bnb.fit(X_train, y_train)
y_pred_bnb   = melhor_bnb.predict(X_test)
y_proba_bnb  = melhor_bnb.predict_proba(X_test)[:, 1]
exibir_resultados_holdout(y_test, y_pred_bnb, y_proba_bnb, 'BernoulliNB')

---
# 8. Comparação dos Modelos

In [ ]:
# Tabela comparativa — Hold-Out
resultados = {
    'KNN': {
        'Acurácia':  accuracy_score(y_test, y_pred_knn),
        'Precisão':  precision_score(y_test, y_pred_knn),
        'Recall':    recall_score(y_test, y_pred_knn),
        'F1':        f1_score(y_test, y_pred_knn),
        'Kappa':     cohen_kappa_score(y_test, y_pred_knn),
        'AUC-ROC':   roc_auc_score(y_test, y_proba_knn)
    },
    'MLP': {
        'Acurácia':  accuracy_score(y_test, y_pred_mlp),
        'Precisão':  precision_score(y_test, y_pred_mlp),
        'Recall':    recall_score(y_test, y_pred_mlp),
        'F1':        f1_score(y_test, y_pred_mlp),
        'Kappa':     cohen_kappa_score(y_test, y_pred_mlp),
        'AUC-ROC':   roc_auc_score(y_test, y_proba_mlp)
    },
    'BernoulliNB': {
        'Acurácia':  accuracy_score(y_test, y_pred_bnb),
        'Precisão':  precision_score(y_test, y_pred_bnb),
        'Recall':    recall_score(y_test, y_pred_bnb),
        'F1':        f1_score(y_test, y_pred_bnb),
        'Kappa':     cohen_kappa_score(y_test, y_pred_bnb),
        'AUC-ROC':   roc_auc_score(y_test, y_proba_bnb)
    }
}

df_resultados = pd.DataFrame(resultados).T

# Destacar melhor valor por coluna
print('=== Comparação dos Modelos — Hold-Out (30%) ===')
display(df_resultados.round(4).style.highlight_max(axis=0, color='#C8E6C9'))

In [ ]:
# Tabela comparativa — K-Fold (média ± desvio)
metricas_nomes = ['acuracia', 'precisao', 'recall', 'f1', 'kappa', 'auc']
modelos_scores = {
    'KNN':        scores_knn_cv,
    'MLP':        scores_mlp_cv,
    'BernoulliNB': scores_bnb_cv
}

print('=== Comparação dos Modelos — K-Fold (5 folds): média ± desvio ===')
linhas = []
for nome, scores in modelos_scores.items():
    linha = {'Modelo': nome}
    for m in metricas_nomes:
        vals = scores[f'test_{m}']
        linha[m.capitalize()] = f'{np.mean(vals):.4f} ± {np.std(vals):.4f}'
    linhas.append(linha)

df_cv = pd.DataFrame(linhas).set_index('Modelo')
display(df_cv)

In [ ]:
# Gráfico comparativo de métricas (Hold-Out)
metricas_plot = ['Acurácia', 'Precisão', 'Recall', 'F1', 'Kappa', 'AUC-ROC']
modelos_plot  = ['KNN', 'MLP', 'BernoulliNB']
cores = ['#2196F3', '#4CAF50', '#FF9800']

x = np.arange(len(metricas_plot))
largura = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for i, (modelo, cor) in enumerate(zip(modelos_plot, cores)):
    valores = [df_resultados.loc[modelo, m] for m in metricas_plot]
    bars = ax.bar(x + i * largura, valores, largura, label=modelo,
                  color=cor, edgecolor='black', linewidth=0.6)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + largura)
ax.set_xticklabels(metricas_plot)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Valor')
ax.set_title('Comparação de Métricas — Hold-Out (30%)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparacao_metricas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrizes de Confusão — 3 modelos lado a lado
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (y_pred_knn, 'KNN'),
    (y_pred_mlp, 'MLP'),
    (y_pred_bnb, 'BernoulliNB')
]

for ax, (y_pred, nome) in zip(axes, configs):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['LGG', 'GBM'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}', fontsize=12, fontweight='bold')

plt.suptitle('Matrizes de Confusão — Hold-Out (30%)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('matrizes_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Curvas ROC — 3 modelos no mesmo gráfico
fig, ax = plt.subplots(figsize=(7, 6))

configs_roc = [
    (y_proba_knn,  'KNN',         '#2196F3'),
    (y_proba_mlp,  'MLP',         '#4CAF50'),
    (y_proba_bnb,  'BernoulliNB', '#FF9800')
]

for y_proba, nome, cor in configs_roc:
    auc_val = roc_auc_score(y_test, y_proba)
    RocCurveDisplay.from_predictions(
        y_test, y_proba,
        name=f'{nome} (AUC={auc_val:.4f})',
        color=cor, ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', label='Baseline aleatório (AUC=0.50)')
ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 9. Análise de Erros — Diferencial do Trabalho

Investigamos **onde** cada modelo erra e se os erros se concentram em pacientes com ausência das mutações moleculares mais informativas para distinguir LGG de GBM (IDH1, TP53, ATRX).

In [ ]:
# Coletar erros de cada modelo no conjunto de teste
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = pd.Series(y_test).reset_index(drop=True)

def coletar_erros(y_true, y_pred, nome):
    mascara = y_true.values != y_pred
    df_erros = X_test_reset[mascara].copy()
    df_erros['real']    = le.inverse_transform(y_true.values[mascara])
    df_erros['predito'] = le.inverse_transform(y_pred[mascara])
    df_erros['modelo']  = nome
    return df_erros

erros_knn = coletar_erros(y_test_reset, y_pred_knn, 'KNN')
erros_mlp = coletar_erros(y_test_reset, y_pred_mlp, 'MLP')
erros_bnb = coletar_erros(y_test_reset, y_pred_bnb, 'BernoulliNB')

print('=== Total de Erros por Modelo ===')
print(f'KNN:        {len(erros_knn)} erros ({len(erros_knn)/len(y_test)*100:.1f}%)')
print(f'MLP:        {len(erros_mlp)} erros ({len(erros_mlp)/len(y_test)*100:.1f}%)')
print(f'BernoulliNB:{len(erros_bnb)} erros ({len(erros_bnb)/len(y_test)*100:.1f}%)')

In [ ]:
# Tipo de erro: FP (LGG→GBM) e FN (GBM→LGG)
def tipo_erro(df_erros, nome):
    fp = ((df_erros['real'] == 'LGG') & (df_erros['predito'] == 'GBM')).sum()
    fn = ((df_erros['real'] == 'GBM') & (df_erros['predito'] == 'LGG')).sum()
    total = len(df_erros)
    print(f'{nome:12s} | Total: {total:3d} | FP (LGG→GBM): {fp:3d} | FN (GBM→LGG): {fn:3d}')
    return fp, fn

print('=== Tipo de Erro por Modelo ===')
print('Nota: FN (GBM→LGG) é clinicamente mais grave — tumor agressivo não detectado')
print('-' * 65)
fp_knn, fn_knn = tipo_erro(erros_knn, 'KNN')
fp_mlp, fn_mlp = tipo_erro(erros_mlp, 'MLP')
fp_bnb, fn_bnb = tipo_erro(erros_bnb, 'BernoulliNB')

In [ ]:
# Análise de erros por feature molecular chave
# Features mais informativas segundo a literatura (Tasci et al., 2022)
FEATURES_CHAVE = ['IDH1', 'TP53', 'ATRX', 'PTEN', 'EGFR']

# Proporção de erros onde a feature está NOT mutated (wildtype)
def taxa_erro_wildtype(df_erros, features):
    resultado = {}
    for feat in features:
        if feat in df_erros.columns:
            # Após o pré-processamento, binárias são 0/1
            # 0 = not_mutated (wildtype)
            taxa = (df_erros[feat] == 0).mean()
            resultado[feat] = taxa
    return resultado

taxa_knn = taxa_erro_wildtype(erros_knn, FEATURES_CHAVE)
taxa_mlp = taxa_erro_wildtype(erros_mlp, FEATURES_CHAVE)
taxa_bnb = taxa_erro_wildtype(erros_bnb, FEATURES_CHAVE)

# Taxa geral no conjunto de teste (baseline de comparação)
taxa_geral = {feat: (X_test_reset[feat] == 0).mean() for feat in FEATURES_CHAVE}

df_analise_erros = pd.DataFrame({
    'Geral (teste)': taxa_geral,
    'KNN (erros)':   taxa_knn,
    'MLP (erros)':   taxa_mlp,
    'BernoulliNB (erros)': taxa_bnb
}).T

print('=== Proporção de Wildtype (não mutado) nos Erros vs. Geral ===')
print('Valores acima da linha "Geral" indicam concentração de erros em wildtypes')
display(df_analise_erros.round(3))

In [ ]:
# Visualização da análise de erros
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(FEATURES_CHAVE))
largura = 0.2
cores_analise = ['#9E9E9E', '#2196F3', '#4CAF50', '#FF9800']
labels_analise = ['Geral (teste)', 'KNN (erros)', 'MLP (erros)', 'BernoulliNB (erros)']

for i, (label_a, cor) in enumerate(zip(labels_analise, cores_analise)):
    valores = df_analise_erros.loc[label_a].values
    ax.bar(x + i * largura, valores, largura,
           label=label_a, color=cor, edgecolor='black', linewidth=0.6)

ax.set_xticks(x + largura * 1.5)
ax.set_xticklabels(FEATURES_CHAVE, fontsize=11)
ax.set_ylabel('Proporção de Wildtype (não mutado)')
ax.set_title('Proporção de Wildtype nos Erros vs. Dataset Geral\n(valores maiores = erros concentrados em wildtypes)',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('analise_erros_molecular.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tabela final consolidada para o artigo
tabela_final = pd.DataFrame({
    'Modelo':        ['KNN', 'MLP', 'BernoulliNB'],
    'Total Erros':   [len(erros_knn), len(erros_mlp), len(erros_bnb)],
    'FP (LGG→GBM)': [fp_knn, fp_mlp, fp_bnb],
    'FN (GBM→LGG)': [fn_knn, fn_mlp, fn_bnb],
    'IDH1 wildtype nos erros (%)': [
        f"{taxa_knn.get('IDH1', 0)*100:.1f}%",
        f"{taxa_mlp.get('IDH1', 0)*100:.1f}%",
        f"{taxa_bnb.get('IDH1', 0)*100:.1f}%"
    ],
    'ATRX wildtype nos erros (%)': [
        f"{taxa_knn.get('ATRX', 0)*100:.1f}%",
        f"{taxa_mlp.get('ATRX', 0)*100:.1f}%",
        f"{taxa_bnb.get('ATRX', 0)*100:.1f}%"
    ]
})

print('=== Tabela Consolidada de Erros (para o artigo) ===')
display(tabela_final)

---
# 10. Resumo Final

In [ ]:
print('=' * 60)
print('RESUMO FINAL — CLASSIFICAÇÃO DE GLIOMAS (TCGA)')
print('=' * 60)

print('\n--- Hold-Out (30%) ---')
display(df_resultados.round(4))

print('\n--- K-Fold (5 folds) ---')
display(df_cv)

melhor = df_resultados['F1'].idxmax()
print(f'\nMelhor modelo por F1 (Hold-Out): {melhor} ({df_resultados.loc[melhor, "F1"]:.4f})')
print(f'Melhor modelo por AUC  (Hold-Out): {df_resultados["AUC-ROC"].idxmax()}')

print('\n--- Análise de Erros ---')
display(tabela_final)

print('\nNota: FN (GBM classificado como LGG) é clinicamente mais grave,')
print('pois representa um tumor agressivo não detectado.')